# Example 11.2 / 4: Classify a Model Before Choosing a Solver

Companion notebook for **11.2 Nonlinear Models** in the AMPL lab Documentation, section "Classify a Model Before Choosing a Solver". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Classify a Model Before Choosing a Solver

The algebraic class is only the start. Convexity, finite variable bounds, differentiability,
solver capabilities, and termination status decide what the computed result proves.

| Question | Why it matters | Concrete expression | Answer |
|---|---|---|---|
| Are all variable-dependent expressions affine? | If yes, the continuous model is an LP. | `minimize 5*x + 3*y` with `x + y >= 1` | yes: an LP, with `_snnlobjs = 0` and `_snnlcons = 0` in the run below |
| Are all remaining terms quadratic? | The model may be a QP or QCP, but definiteness and constraint direction still decide convexity. | the same objective with `x*y >= 1` | a QCP whose objective is linear: `_snnlcons = 1` below; $xy\ge1$ is not convex over all signs (see Quadratic Models (AMPL lab documentation, section 11.1)) |
| Can each nonlinear relation be expressed by a supported cone? | A recognized conic form lets a conic solver use specialized methods. | `sqrt(sum {i in ITEMS} Mix[i]^2) <= Radius` | a second-order cone; Second-Order Cones (AMPL lab documentation, section 11.5) shows how a solver receives it |
| Are there integer variables? | The problem becomes MIQP, MIQCP, or MINLP; continuous relaxations and discrete search both matter. | `var Open {PLANTS} binary;` with `quad_cost[p] * Produce[p]^2` | an MIQP; see Mixed-Integer Nonlinear Models (AMPL lab documentation, section 11.4.2) |
| Are complementarity pairs present? | A square MCP and an optimization problem with complementarity need different solver interfaces. | `Price >= 0 complements 2*Price >= 6` | one linear complementarity constraint; see Complementarity Constraints (AMPL lab documentation, section 11.6) |
| Are functions defined and differentiable over all variable bounds? | Local derivative-based methods need safe evaluations; global methods also need valid domains and bounds. | `log(1 + Spend[p])` with `Spend[p] >= 0` | always defined, while `log(0)` fails with `Result too large` (see "Domains and Units" above) |

AMPL reports how it classified the problem it sent to the solver through generic names (see
Generic Names, Statistics and Message Limits (AMPL lab documentation, section 6.8.1)): `_snnlobjs` counts the nonlinear
objectives and `_snnlcons` the nonlinear constraints the solver received. Read with
`get_value()`, they change when the linear row `Cover` is replaced by the product row `Product`;
the amplpy `drop()` and `restore()` methods switch a constraint off and on:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
var x >= 0, <= 4;  # first activity level
var y >= 0, <= 4;  # second activity level

minimize Cost: 5 * x + 3 * y;  # linear objective in either model

subject to Cover: x + y >= 1;    # linear covering row
subject to Product: x * y >= 1;  # bilinear row: makes the model nonlinear

Writing model.mod


### amplpy: run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# run.py
names = ("_snnlobjs", "_snnlcons")
ampl.con["Product"].drop()
ampl.solve(solver="highs", verbose=False)
print("Cover only:  ", [ampl.get_value(n) for n in names], round(ampl.obj["Cost"].value(), 6))
ampl.con["Cover"].drop()
ampl.con["Product"].restore()
ampl.solve(solver="ipopt", verbose=False)
print("Product only:", [ampl.get_value(n) for n in names], round(ampl.obj["Cost"].value(), 6))

Cover only:   [0, 0] 3.0
Product only: [0, 1] 7.745967
